In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

raw_path = Path(
    r"C:\Users\Administrator\AppData\Local\Temp\labeled_data.csv"
)
folder = Path(
    r"C:\Users\Administrator\Desktop\kamp"
    r"\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋"
)

raw = pd.read_csv(raw_path)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

for model in ["cn7", "rg3"]:
    scaled = pd.read_csv(folder / f"moldset_labeled_{model}.csv")

    # 첫 번째 컬럼에 저장된 원본 행 인덱스로 연결
    index_col = "Unnamed: 0"
    idx = pd.to_numeric(scaled[index_col], errors="raise")

    if (
        idx.isna().any()
        or (idx % 1 != 0).any()
        or not idx.between(0, len(raw) - 1).all()
        or idx.duplicated().any()
    ):
        raise ValueError(f"{model}: 원본 인덱스가 올바르지 않습니다.")

    source = raw.iloc[idx.astype("int64").to_numpy()].reset_index(drop=True)

    # 원본과 표준화된 값이 일치하는지 재검증
    features = [
        c for c in scaled.columns
        if c not in [index_col, "PassOrFail"]
    ]
    x = source[features].astype(float)
    expected = (x - x.mean()) / x.std(ddof=0).replace(0, 1)

    if not np.allclose(
        expected.to_numpy(),
        scaled[features].to_numpy(dtype=float),
        rtol=1e-7,
        atol=1e-7,
        equal_nan=True,
    ):
        raise ValueError(f"{model}: 표준화 값이 원본과 일치하지 않습니다.")

    if not np.array_equal(
        source["PassOrFail"].map({"Y": 0, "N": 1}).to_numpy(),
        scaled["PassOrFail"].to_numpy(),
    ):
        raise ValueError(f"{model}: PassOrFail이 일치하지 않습니다.")

    if not source["PART_NAME"].str.startswith(model.upper(), na=False).all():
        raise ValueError(f"{model}: 제품명이 일치하지 않습니다.")

    side = source["PART_NAME"].str.extract(r"\b(RH|LH)\s*$", expand=False)
    if side.isna().any():
        raise ValueError(f"{model}: RH/LH를 구분할 수 없는 행이 있습니다.")

    # 기존 측정값과 행 순서를 유지하면서 메타데이터 추가
    result = scaled.rename(columns={index_col: "original_index"}).copy()
    result["TimeStamp"] = source["TimeStamp"].to_numpy()
    result["PART_NAME"] = source["PART_NAME"].to_numpy()
    result["RH_LH"] = side.to_numpy()

    # 실행 시각을 파일명에 붙이고, 같은 이름이 있으면 덮어쓰기 금지
    output = folder / f"moldset_labeled_{model}_with_metadata_{run_id}.csv"
    result.to_csv(output, index=False, encoding="utf-8-sig", mode="x")

    print(f"저장 완료: {output}")
    print(f"행 수: {len(result):,} / RH·LH: {result['RH_LH'].value_counts().to_dict()}")

저장 완료: C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7_with_metadata_20260928_190536_568856.csv
행 수: 1,211 / RH·LH: {'RH': 606, 'LH': 605}
저장 완료: C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3_with_metadata_20260928_190536_568856.csv
행 수: 1,182 / RH·LH: {'LH': 591, 'RH': 591}
